# NIST Cross-Dataset Validation

This notebook:
1. Loads the concatenated NIST neQuant data
2. Preprocesses it through the NFI-trained pipeline (column alignment, O renormalization, CLR, etc.)
3. Runs the tuned MLP on non-shooter samples (specificity test)
4. Runs the tuned MLP on shooter samples (sensitivity contrast)
5. Analyzes results and generates visualizations

In [36]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import pickle
import os
import warnings
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, roc_auc_score

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cpu


## 1. Load NIST Data

In [3]:
nist = pd.read_parquet(
    "../../data/processed/nist_concatenated_parquets/nist_neQuant_concatenated_data.parquet"
)
print(f"NIST shape: {nist.shape}")
print(f"Sample sources: {nist['sample_source'].nunique()}")
print(f"\nSample source counts:")
print(nist["sample_source"].value_counts().to_string())

NIST shape: (135447, 103)
Sample sources: 30

Sample source counts:
sample_source
brakes_chevy_front_driver                      5000
brakes_chevy_front_passenger                   5000
shooter_5L                                     5000
shooter_4R                                     5000
shooter_4L                                     5000
shooter_3R                                     5000
shooter_3L                                     5000
shooter_2                                      5000
shooter_1                                      5000
fireworks_spinners_postignite                  5000
fireworks_spinners_posthandling_preignite      5000
fireworks_spinners_postcleanup                 5000
fireworks_spinners_debris                      5000
fireworks_sparklers_posthandling_postburn      5000
fireworks_sparklers_debris                     5000
fireworks_sparklers_burning                    5000
fireworks_romancandles_posthandle_preignite    5000
fireworks_romancandles_postcleanup

In [4]:
shooter_mask = nist["sample_source"].str.startswith("shooter")
nist_nonshooter = nist[~shooter_mask].copy()
nist_shooter = nist[shooter_mask].copy()

print(
    f"Non-shooter particles: {len(nist_nonshooter):,} ({nist_nonshooter['sample_source'].nunique()} sources)"
)
print(
    f"Shooter particles: {len(nist_shooter):,} ({nist_shooter['sample_source'].nunique()} sources)"
)

print(f"\nNon-shooter CLASS distribution (top 15):")
print(nist_nonshooter["CLASS"].value_counts().head(15).to_string())
print(f"\nShooter CLASS distribution (top 15):")
print(nist_shooter["CLASS"].value_counts().head(15).to_string())

Non-shooter particles: 95,447 (22 sources)
Shooter particles: 40,000 (8 sources)

Non-shooter CLASS distribution (top 15):
CLASS
#Unclassified#        42718
Iron-bearing          18993
Ca-bearing             9842
Iron                   8934
Silicate               4867
Uranium-bearing        2498
Quartz-like            2303
Dolomite               1427
Zinc                   1135
Aluminosilicate         917
Other Silicate          540
Aluminum                302
Calcium silicate        282
Al-Mg alloy (5XXX)      130
Fe-Al alloy             118

Shooter CLASS distribution (top 15):
CLASS
Ca-bearing         16479
#Unclassified#      9958
Silicate            3830
Iron-bearing        2036
Iron                1893
Uranium-bearing     1183
Quartz-like          844
Char GSR-like        694
Dolomite             666
Aluminosilicate      495
Pb-Ba                264
Aluminum             206
Other Silicate       204
Ba-Sb                204
Zinc                 165


## 2. Preprocess 

In [5]:
nfi_elements = [
    "o",
    "s",
    "cu",
    "ba",
    "al",
    "si",
    "ca",
    "pb",
    "sb",
    "fe",
    "zn",
    "cl",
    "k",
    "na",
    "mg",
    "ti",
    "sn",
    "p",
    "mn",
    "as",
    "cr",
    "br",
    "mo",
    "sr",
    "ni",
    "w",
    "hg",
]

nist_element_cols_upper = [
    "O",
    "MG",
    "AL",
    "SI",
    "P",
    "S",
    "CL",
    "K",
    "CA",
    "TI",
    "CR",
    "MN",
    "FE",
    "NI",
    "CU",
    "ZN",
    "RB",
    "SR",
    "ZR",
    "MO",
    "RH",
    "PD",
    "AG",
    "IN",
    "SN",
    "SB",
    "BA",
    "LA",
    "CE",
    "ND",
    "SM",
    "TB",
    "TM",
    "AU",
    "PB",
    "BI",
]

missing_from_nist = ["na", "as", "br", "w", "hg"]
print(f"NFI elements missing from NIST (will zero-fill): {missing_from_nist}")

nist_only = [e for e in nist_element_cols_upper if e.lower() not in nfi_elements]
print(f"NIST-only elements (will drop): {nist_only}")

NFI elements missing from NIST (will zero-fill): ['na', 'as', 'br', 'w', 'hg']
NIST-only elements (will drop): ['RB', 'ZR', 'RH', 'PD', 'AG', 'IN', 'LA', 'CE', 'ND', 'SM', 'TB', 'TM', 'AU', 'BI']


In [6]:
def preprocess_nist(df):
    result = pd.DataFrame()
    sum_with_o = df["sum_with_oxygen"].values

    for nist_col in nist_element_cols_upper:
        nfi_name = nist_col.lower()
        if nfi_name in nfi_elements:
            result[nfi_name] = (df[nist_col].values / sum_with_o) * 100.0

    for elem in missing_from_nist:
        result[elem] = 0.0

    result = result[nfi_elements]
    result["sample_source"] = df["sample_source"].values
    result["CLASS"] = df["CLASS"].values
    return result


nist_nonshooter_proc = preprocess_nist(nist_nonshooter)
nist_shooter_proc = preprocess_nist(nist_shooter)

print(f"Non-shooter processed shape: {nist_nonshooter_proc.shape}")
print(f"Shooter processed shape: {nist_shooter_proc.shape}")

elem_sums = nist_nonshooter_proc[nfi_elements].sum(axis=1)
print(f"\nNon-shooter element sum stats:")
print(
    f"  Mean: {elem_sums.mean():.2f}, Std: {elem_sums.std():.4f}, Min: {elem_sums.min():.2f}, Max: {elem_sums.max():.2f}"
)

Non-shooter processed shape: (95447, 29)
Shooter processed shape: (40000, 29)

Non-shooter element sum stats:
  Mean: 91.52, Std: 7.7408, Min: 10.01, Max: 100.00


## 3. Feature Engineering 
Apply the same CLR transform, binary presence indicators, and engineered featuresused for the NFI training data. All scalers/parameters from NFI training only.

In [7]:
def multiplicative_replacement(X, delta=None):
    X = X.copy().astype(float)
    if delta is None:
        nonzero_vals = X[X > 0]
        delta = nonzero_vals.min() / 2 if len(nonzero_vals) > 0 else 1e-10
    row_sums = X.sum(axis=1, keepdims=True)
    for i in range(X.shape[0]):
        zero_mask = X[i] == 0
        n_zeros = zero_mask.sum()
        if n_zeros > 0 and row_sums[i, 0] > 0:
            X[i, zero_mask] = delta
            correction = 1 - (n_zeros * delta / row_sums[i, 0])
            X[i, ~zero_mask] *= correction
    return X


def clr_transform(X):
    log_X = np.log(X)
    geometric_mean = log_X.mean(axis=1, keepdims=True)
    return log_X - geometric_mean


def engineer_nn_features(df, element_cols):
    X_raw = df[element_cols].values
    X_replaced = multiplicative_replacement(X_raw)
    X_clr = clr_transform(X_replaced).astype(np.float32)
    X_presence = (X_raw > 0).astype(np.float32)

    total_mass = df[element_cols].sum(axis=1).values
    denom = total_mass - df["ba"].values - df["o"].values
    safe_denom = np.where(denom == 0, -1, denom)
    pb_sb_ratio = ((df["pb"].values + df["sb"].values) / safe_denom).astype(np.float32)
    log_pb_sb = np.log1p(df["pb"].values + df["sb"].values).astype(np.float32)

    X_features = np.hstack(
        [X_clr, X_presence, pb_sb_ratio.reshape(-1, 1), log_pb_sb.reshape(-1, 1)]
    )
    return X_features

In [8]:
X_nonshooter = engineer_nn_features(nist_nonshooter_proc, nfi_elements)
X_shooter = engineer_nn_features(nist_shooter_proc, nfi_elements)

print(f"Non-shooter feature matrix: {X_nonshooter.shape}")
print(f"Shooter feature matrix: {X_shooter.shape}")
print(f"Expected features: 56")
print(
    f"\nNon-shooter NaN: {np.isnan(X_nonshooter).sum()}, Inf: {np.isinf(X_nonshooter).sum()}"
)
print(f"Shooter NaN: {np.isnan(X_shooter).sum()}, Inf: {np.isinf(X_shooter).sum()}")

Non-shooter feature matrix: (95447, 56)
Shooter feature matrix: (40000, 56)
Expected features: 56

Non-shooter NaN: 0, Inf: 0
Shooter NaN: 0, Inf: 0


## 4. Load Tuned Model & Scaler

In [9]:
artifacts_dir = "../../artifacts/models/neural_network"
with open(os.path.join(artifacts_dir, "scaler.pkl"), "rb") as f:
    scaler = pickle.load(f)

X_nonshooter_scaled = scaler.transform(X_nonshooter)
X_shooter_scaled = scaler.transform(X_shooter)
print("Scaled with NFI-fitted scaler.")

Scaled with NFI-fitted scaler.


In [26]:
# compare NFI vs NIST feature distributions

df_nfi = pd.read_parquet("../../data/processed/engineered_features_nn.parquet")
if "gsr_count" in df_nfi.columns:
    df_nfi = df_nfi.drop(columns=["gsr_count"])

meta_cols = ["stub_id", "particle_id", "label", "target", "final_class"]
feature_cols = [c for c in df_nfi.columns if c not in meta_cols]

# sample of NFI data (same size as NIST non-shooter for fair comparison)
nfi_sample = df_nfi.sample(
    n=min(len(X_nonshooter_scaled), len(df_nfi)), random_state=42
)
X_nfi_sample = nfi_sample[feature_cols].values.astype(np.float32)
X_nfi_scaled = scaler.transform(X_nfi_sample)

print("scaled feature range comparison")
print(
    f"{'Feature':<35} {'NFI min':>10} {'NFI max':>10} {'NIST min':>10} {'NIST max':>10} {'NFI mean':>10} {'NIST mean':>10}"
)
print("-" * 105)
for i, col in enumerate(feature_cols):
    nfi_min = X_nfi_scaled[:, i].min()
    nfi_max = X_nfi_scaled[:, i].max()
    nist_min = X_nonshooter_scaled[:, i].min()
    nist_max = X_nonshooter_scaled[:, i].max()
    nfi_mean = X_nfi_scaled[:, i].mean()
    nist_mean = X_nonshooter_scaled[:, i].mean()
    flag = " <<<" if abs(nfi_mean - nist_mean) > 2.0 else ""
    print(
        f"{col:<35} {nfi_min:>10.2f} {nfi_max:>10.2f} {nist_min:>10.2f} {nist_max:>10.2f} {nfi_mean:>10.2f} {nist_mean:>10.2f}{flag}"
    )

scaled feature range comparison
Feature                                NFI min    NFI max   NIST min   NIST max   NFI mean  NIST mean
---------------------------------------------------------------------------------------------------------
clr_o                                   -12.65       2.58     -12.90       4.23      -0.01      -1.27
clr_s                                    -2.83       1.39      -3.71       1.69      -0.02       0.05
clr_cu                                   -1.99       1.63      -2.89       1.70      -0.03      -0.35
clr_ba                                   -2.13       1.38      -2.86       1.73       0.01      -0.23
clr_al                                   -1.83       1.82      -2.75       2.14       0.03      -0.37
clr_si                                   -1.64       1.92      -2.61       2.58       0.06       0.07
clr_ca                                   -1.61       2.26      -2.50       2.74       0.06       0.62
clr_pb                                   -1.43

In [27]:
nfi_means = X_nfi_scaled.mean(axis=0)
nist_means = X_nonshooter_scaled.mean(axis=0)
mean_diffs = np.abs(nfi_means - nist_means)
print(
    f"Features with mean difference > 2 std: {(mean_diffs > 2.0).sum()} / {len(feature_cols)}"
)
print(
    f"Features with mean difference > 5 std: {(mean_diffs > 5.0).sum()} / {len(feature_cols)}"
)
print(
    f"Max mean difference: {mean_diffs.max():.2f} (feature: {feature_cols[mean_diffs.argmax()]})"
)
print(f"Average mean difference: {mean_diffs.mean():.2f}")

Features with mean difference > 2 std: 9 / 56
Features with mean difference > 5 std: 1 / 56
Max mean difference: 7.68 (feature: has_sr)
Average mean difference: 1.12


## This is problematic...

## 4.1 Re-engineer with fair features 

## Retrain MLP using only the 22 elements that exist in **both** NFI and NIST (drop Na, As, Br, W, Hg from the NFI pipeline entirely), then apply that model to NIST, that way both datasets are on equal footing.

In [28]:
shared_elements = [e for e in nfi_elements if e not in ["na", "as", "br", "w", "hg"]]
print(f"Shared elements ({len(shared_elements)}): {shared_elements}")

Shared elements (22): ['o', 's', 'cu', 'ba', 'al', 'si', 'ca', 'pb', 'sb', 'fe', 'zn', 'cl', 'k', 'mg', 'ti', 'sn', 'p', 'mn', 'cr', 'mo', 'sr', 'ni']


In [30]:
df_nfi_raw = pd.read_parquet("../../data/processed/preprocessed.parquet")
meta_cols_raw = [
    "stub_id",
    "particle_id",
    "label",
    "target",
    "final_class",
    "relevance_class",
    "merged_relevance_class",
]
nfi_shared = df_nfi_raw[
    [c for c in df_nfi_raw.columns if c in shared_elements or c in meta_cols_raw]
].copy()
nfi_shared = nfi_shared[nfi_shared["label"].isin(["GSR", "Non_GSR"])].copy()
nfi_shared["target"] = (nfi_shared["label"] == "GSR").astype(float)

print(f"NFI shared-element dataset: {nfi_shared.shape}")
print(f"Elements: {[c for c in nfi_shared.columns if c in shared_elements]}")

NFI shared-element dataset: (2294985, 29)
Elements: ['o', 's', 'cu', 'ba', 'al', 'si', 'ca', 'pb', 'sb', 'fe', 'zn', 'cl', 'k', 'mg', 'ti', 'sn', 'p', 'mn', 'cr', 'mo', 'sr', 'ni']


In [31]:
X_nfi_shared = engineer_nn_features(nfi_shared, shared_elements)
y_nfi = nfi_shared["target"].values.astype(np.float32)
groups_nfi = nfi_shared["stub_id"].values

print(
    f"NFI shared feature matrix: {X_nfi_shared.shape}"
)  # should be 22 CLR + 22 presence + 2 engineered = 46 features

NFI shared feature matrix: (2294985, 46)


In [34]:
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
trainval_idx, test_idx = next(gss1.split(X_nfi_shared, y_nfi, groups_nfi))

X_trainval = X_nfi_shared[trainval_idx]
y_trainval = y_nfi[trainval_idx]
groups_trainval = groups_nfi[trainval_idx]

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx_rel, val_idx_rel = next(gss2.split(X_trainval, y_trainval, groups_trainval))

X_train_raw = X_trainval[train_idx_rel]
X_val_raw = X_trainval[val_idx_rel]
y_train = y_trainval[train_idx_rel]
y_val = y_trainval[val_idx_rel]

X_test_nfi_raw = X_nfi_shared[test_idx]
y_test_nfi = y_nfi[test_idx]

scaler_shared = StandardScaler()
X_train_s = scaler_shared.fit_transform(X_train_raw)
X_val_s = scaler_shared.transform(X_val_raw)
X_test_nfi_s = scaler_shared.transform(X_test_nfi_raw)

print(f"Train: {len(X_train_s):,}, Val: {len(X_val_s):,}, Test: {len(X_test_nfi_s):,}")

Train: 1,444,147, Val: 407,614, Test: 443,224


## Tunable MLP

In [35]:
class TunableMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, n_layers=2, dropout=0.3):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for _ in range(n_layers):
            layers.extend(
                [
                    nn.Linear(prev_dim, hidden_dim),
                    nn.ReLU(),
                    nn.Dropout(dropout),
                ]
            )
            prev_dim = hidden_dim
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [37]:
model_shared = TunableMLP(
    input_dim=X_train_s.shape[1], hidden_dim=256, n_layers=2, dropout=0.3
).to(device)

optimizer = torch.optim.Adam(model_shared.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()

train_ds = TensorDataset(
    torch.tensor(X_train_s, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32),
)
val_ds = TensorDataset(
    torch.tensor(X_val_s, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32)
)
train_loader = DataLoader(train_ds, batch_size=4096, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=4096, shuffle=False)

best_val_loss = float("inf")
best_state = None
patience_counter = 0

for epoch in range(20):
    model_shared.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        loss = criterion(model_shared(xb).squeeze(), yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    model_shared.eval()
    val_loss = 0
    n_val = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            val_loss += criterion(model_shared(xb).squeeze(), yb).item()
            n_val += 1
    vl = val_loss / n_val
    print(f"Epoch {epoch + 1}: val_loss={vl:.6f}")

    if vl < best_val_loss:
        best_val_loss = vl
        best_state = {k: v.clone() for k, v in model_shared.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= 5:
            break

model_shared.load_state_dict(best_state)
print(f"\nBest val loss: {best_val_loss:.6f}")

Epoch 1: val_loss=0.005075
Epoch 2: val_loss=0.004358
Epoch 3: val_loss=0.003766
Epoch 4: val_loss=0.003193
Epoch 5: val_loss=0.002969
Epoch 6: val_loss=0.002560
Epoch 7: val_loss=0.002569
Epoch 8: val_loss=0.002219
Epoch 9: val_loss=0.002447
Epoch 10: val_loss=0.002003
Epoch 11: val_loss=0.002112
Epoch 12: val_loss=0.001992
Epoch 13: val_loss=0.002022
Epoch 14: val_loss=0.001931
Epoch 15: val_loss=0.001844
Epoch 16: val_loss=0.001758
Epoch 17: val_loss=0.001613
Epoch 18: val_loss=0.001659
Epoch 19: val_loss=0.001892
Epoch 20: val_loss=0.001662

Best val loss: 0.001613


In [38]:
model_shared.eval()
with torch.no_grad():
    nfi_logits = (
        model_shared(torch.tensor(X_test_nfi_s, dtype=torch.float32).to(device))
        .squeeze()
        .cpu()
        .numpy()
    )
nfi_probs = 1 / (1 + np.exp(-nfi_logits))
nfi_preds = (nfi_probs >= 0.5).astype(int)
cm_nfi = confusion_matrix(y_test_nfi, nfi_preds)
print(
    f"\nNFI Test: TN={cm_nfi[0, 0]:,} FP={cm_nfi[0, 1]:,} FN={cm_nfi[1, 0]:,} TP={cm_nfi[1, 1]:,}"
)
print(f"NFI Test ROC-AUC: {roc_auc_score(y_test_nfi, nfi_probs):.4f}")
print(f"NFI Test FPR: {cm_nfi[0, 1] / cm_nfi[0].sum():.4f}")


NFI Test: TN=233,061 FP=214 FN=51 TP=209,898
NFI Test ROC-AUC: 1.0000
NFI Test FPR: 0.0009


In [40]:
X_nist_ns_shared = engineer_nn_features(nist_nonshooter_proc, shared_elements)
X_nist_sh_shared = engineer_nn_features(nist_shooter_proc, shared_elements)

X_nist_ns_s = scaler_shared.transform(X_nist_ns_shared)
X_nist_sh_s = scaler_shared.transform(X_nist_sh_shared)

ns_probs = predict_probs(model_shared, X_nist_ns_s)
sh_probs = predict_probs(model_shared, X_nist_sh_s)

print("Non-Shooter")
for t in [0.5, 0.9]:
    preds = (ns_probs >= t).astype(int)
    n_fp = preds.sum()
    print(
        f"  Threshold={t}: {n_fp:,} FP out of {len(preds):,} ({n_fp / len(preds):.2%})"
    )

print("Shooter")
for t in [0.5, 0.9]:
    preds = (sh_probs >= t).astype(int)
    n_gsr = preds.sum()
    print(
        f"  Threshold={t}: {n_gsr:,} GSR out of {len(preds):,} ({n_gsr / len(preds):.2%})"
    )

print(f"Diff")
ns_rate = (ns_probs >= 0.5).mean()
sh_rate = (sh_probs >= 0.5).mean()
print(f"Non-shooter GSR rate: {ns_rate:.2%}")
print(f"Shooter GSR rate: {sh_rate:.2%}")
print(f"Ratio: {sh_rate / max(ns_rate, 1e-10):.1f}x")

Non-Shooter
  Threshold=0.5: 70,328 FP out of 95,447 (73.68%)
  Threshold=0.9: 65,299 FP out of 95,447 (68.41%)
Shooter
  Threshold=0.5: 25,376 GSR out of 40,000 (63.44%)
  Threshold=0.9: 24,086 GSR out of 40,000 (60.21%)
Diff
Non-shooter GSR rate: 73.68%
Shooter GSR rate: 63.44%
Ratio: 0.9x


## 5. Run Model on Non-Shooter Data (Specificity Test)
All non-shooter particles should be Non-GSR. Any GSR prediction is a false positive.

In [11]:
def predict_probs(model, X_scaled, batch_size=4096):
    model.eval()
    loader = DataLoader(
        TensorDataset(torch.tensor(X_scaled, dtype=torch.float32)),
        batch_size=batch_size,
        shuffle=False,
    )
    all_probs = []
    with torch.no_grad():
        for (xb,) in loader:
            xb = xb.to(device)
            logits = model(xb).squeeze().cpu().numpy()
            probs = 1 / (1 + np.exp(-logits))
            all_probs.append(probs)
    return np.concatenate(all_probs)

In [41]:
nonshooter_probs = predict_probs(model_shared, X_nist_ns_s)
print(f"Non-shooter predictions computed: {len(nonshooter_probs):,} particles")

Non-shooter predictions computed: 95,447 particles


In [42]:
for name, thresh in thresholds.items():
    preds = (nonshooter_probs >= thresh).astype(int)
    n_fp = preds.sum()
    fpr = n_fp / len(preds)
    print(
        f"Threshold={thresh:.4f} ({name}): {n_fp:,} false positives out of {len(preds):,} ({fpr:.4%})"
    )

Threshold=0.5000 (default): 70,328 false positives out of 95,447 (73.6828%)
Threshold=0.4316 (balanced): 70,668 false positives out of 95,447 (74.0390%)
Threshold=0.9830 (high_spec): 58,706 false positives out of 95,447 (61.5064%)
Threshold=0.6134 (high_sens): 69,659 false positives out of 95,447 (72.9819%)


In [43]:
sources = nist_nonshooter_proc["sample_source"].values
nonshooter_preds = (nonshooter_probs >= 0.5).astype(int)

source_results = []
for src in sorted(np.unique(sources)):
    mask = sources == src
    n = mask.sum()
    n_fp = nonshooter_preds[mask].sum()
    mean_p = nonshooter_probs[mask].mean()
    source_type = "brakes" if "brakes" in src else "fireworks"
    source_results.append(
        {
            "Source": src,
            "Type": source_type,
            "N": n,
            "FP": n_fp,
            "FPR": f"{n_fp / n:.4%}",
            "Mean P(GSR)": f"{mean_p:.6f}",
        }
    )

print(pd.DataFrame(source_results).to_string(index=False))

for stype in ["brakes", "fireworks"]:
    mask = np.array(
        [("brakes" in s if stype == "brakes" else "fireworks" in s) for s in sources]
    )
    n = mask.sum()
    n_fp = nonshooter_preds[mask].sum()
    print(f"\n{stype.upper()} total: {n_fp:,} FP out of {n:,} ({n_fp / n:.4%})")

                                     Source      Type    N   FP      FPR Mean P(GSR)
                  brakes_chevy_front_driver    brakes 5000 2314 46.2800%    0.451007
               brakes_chevy_front_passenger    brakes 5000 2278 45.5600%    0.446451
                   brakes_chevy_rear_driver    brakes 5000 4863 97.2600%    0.967439
                brakes_chevy_rear_passenger    brakes 5000 4864 97.2800%    0.968504
              brakes_ford_a213_front_driver    brakes 5000 4523 90.4600%    0.889654
           brakes_ford_a213_front_passenger    brakes 5000 4356 87.1200%    0.855967
               brakes_ford_a213_rear_driver    brakes 5000 4553 91.0600%    0.897148
            brakes_ford_a213_rear_passenger    brakes 5000 4446 88.9200%    0.876859
              brakes_ford_b297_front_driver    brakes 5000 4021 80.4200%    0.793103
           brakes_ford_b297_front_passenger    brakes 5000 4586 91.7200%    0.905985
               brakes_ford_b297_rear_driver    brakes  100   43 4

In [44]:
if nonshooter_preds.sum() > 0:
    print("False positive by class breakdown")
    fp_mask = nonshooter_preds == 1
    fp_classes = nist_nonshooter_proc["CLASS"].values[fp_mask]
    fp_class_counts = pd.Series(fp_classes).value_counts()
    print(fp_class_counts.to_string())
else:
    print("\nNo false positives in non-shooter data!")

False positive by class breakdown
#Unclassified#        36227
Iron-bearing          13001
Ca-bearing             6995
Iron                   5013
Silicate               3037
Uranium-bearing        1947
Dolomite                972
Quartz-like             887
Aluminosilicate         574
Zinc                    571
Other Silicate          356
Calcium silicate        202
Aluminum                163
Al-Mg alloy (5XXX)       75
Ca-Fe silicate           71
Fe-Al alloy              59
Ca-Ti silicate           45
Olivine                  30
Cu-rich                  24
Al-Fe alloy              24
Fe-Cr-Ni Stainless       12
Pyrite (FeS2)            11
Titanium                  8
Fe-Cr stainless           4
Al-Cu alloy (2XXX)        4
Willemite                 3
Cr-bearing                3
Nickel                    3
Cupronickel               2
Brass                     2
Sodium sulfide            1
Al-Zn alloy (7XXX)        1
MoS2                      1


## 6. Run Model on Shooter Data (Sensitivity Contrast)

In [45]:
shooter_probs = predict_probs(model_shared, X_nist_sh_s)
print(f"Shooter predictions computed: {len(shooter_probs):,} particles")

Shooter predictions computed: 40,000 particles


In [46]:
for name, thresh in thresholds.items():
    preds = (shooter_probs >= thresh).astype(int)
    n_gsr = preds.sum()
    gsr_rate = n_gsr / len(preds)
    print(
        f"Threshold={thresh:.4f} ({name}): {n_gsr:,} classified as GSR out of {len(preds):,} ({gsr_rate:.2%})"
    )

Threshold=0.5000 (default): 25,376 classified as GSR out of 40,000 (63.44%)
Threshold=0.4316 (balanced): 25,479 classified as GSR out of 40,000 (63.70%)
Threshold=0.9830 (high_spec): 22,099 classified as GSR out of 40,000 (55.25%)
Threshold=0.6134 (high_sens): 25,177 classified as GSR out of 40,000 (62.94%)


In [47]:
print("GSR rate by source (threshold=0.5)")
shooter_sources = nist_shooter_proc["sample_source"].values
shooter_preds = (shooter_probs >= 0.5).astype(int)

shooter_results = []
for src in sorted(np.unique(shooter_sources)):
    mask = shooter_sources == src
    n = mask.sum()
    n_gsr = shooter_preds[mask].sum()
    mean_p = shooter_probs[mask].mean()
    shooter_results.append(
        {
            "Source": src,
            "N": n,
            "GSR": n_gsr,
            "GSR Rate": f"{n_gsr / n:.2%}",
            "Mean P(GSR)": f"{mean_p:.4f}",
        }
    )

print(pd.DataFrame(shooter_results).to_string(index=False))

GSR rate by source (threshold=0.5)
    Source    N  GSR GSR Rate Mean P(GSR)
 shooter_1 5000 3185   63.70%      0.6346
 shooter_2 5000 3028   60.56%      0.6000
shooter_3L 5000 3314   66.28%      0.6567
shooter_3R 5000 3291   65.82%      0.6521
shooter_4L 5000 3046   60.92%      0.6047
shooter_4R 5000 2998   59.96%      0.5961
shooter_5L 5000 3174   63.48%      0.6308
shooter_5R 5000 3340   66.80%      0.6638


In [48]:
print("GSR rate by NIST class (threshold=0.5)")
shooter_classes = nist_shooter_proc["CLASS"].values
class_results = []
for cls in sorted(np.unique(shooter_classes)):
    mask = shooter_classes == cls
    n = mask.sum()
    if n < 10:
        continue
    n_gsr = shooter_preds[mask].sum()
    mean_p = shooter_probs[mask].mean()
    class_results.append(
        {
            "CLASS": cls,
            "N": n,
            "GSR": n_gsr,
            "GSR Rate": f"{n_gsr / n:.2%}",
            "Mean P(GSR)": f"{mean_p:.4f}",
        }
    )

print(pd.DataFrame(class_results).to_string(index=False))

GSR rate by NIST class (threshold=0.5)
             CLASS     N   GSR GSR Rate Mean P(GSR)
    #Unclassified#  9958  6524   65.52%      0.6467
   Aluminosilicate   495   204   41.21%      0.4028
          Aluminum   206    75   36.41%      0.3585
             Ba-Sb   204   203   99.51%      0.9950
             Brass    81    43   53.09%      0.5128
        Ca-bearing 16479 11118   67.47%      0.6767
  Calcium silicate    68    36   52.94%      0.5311
     Char GSR-like   694   694  100.00%      1.0000
        Cr-bearing    26    10   38.46%      0.3741
           Cu-rich   136   102   75.00%      0.7406
          Dolomite   666   438   65.77%      0.6594
   Fe-Cr stainless    34     8   23.53%      0.2519
Fe-Cr-Ni Stainless   157    54   34.39%      0.3388
              Iron  1893   871   46.01%      0.4445
      Iron-bearing  2036  1225   60.17%      0.5895
              Lead    44    42   95.45%      0.9370
            Nickel    11     4   36.36%      0.3637
           Olivine    14 

## 7. Probability Distribution Comparison
Comparing P(GSR) distributions between non-shooter and shooter.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(
    nonshooter_probs,
    bins=100,
    alpha=0.7,
    label="Non-shooter",
    color="blue",
    density=True,
)
axes[0].hist(
    shooter_probs, bins=100, alpha=0.7, label="Shooter", color="red", density=True
)
axes[0].set_xlabel("P(GSR)")
axes[0].set_ylabel("Density")
axes[0].set_title("P(GSR) Distribution: Non-Shooter vs Shooter")
axes[0].legend()
axes[0].set_yscale("log")
axes[0].grid(True, alpha=0.3)

axes[1].hist(nonshooter_probs, bins=100, alpha=0.7, color="blue", edgecolor="black")
axes[1].set_xlabel("P(GSR)")
axes[1].set_ylabel("Count")
axes[1].set_title("Non-Shooter P(GSR) Distribution")
axes[1].axvline(x=0.5, color="red", linestyle="--", alpha=0.7, label="Threshold=0.5")
axes[1].legend()
axes[1].set_yscale("log")
axes[1].grid(True, alpha=0.3)

axes[2].hist(shooter_probs, bins=100, alpha=0.7, color="red", edgecolor="black")
axes[2].set_xlabel("P(GSR)")
axes[2].set_ylabel("Count")
axes[2].set_title("Shooter P(GSR) Distribution")
axes[2].axvline(x=0.5, color="red", linestyle="--", alpha=0.7, label="Threshold=0.5")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(
    "outputs/NN_nist_validation__p_gsr_distribution_non_shooter_vs_shooter.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

In [50]:
print("Summary stats")
print(f"\nNon-shooter:")
print(f"N: {len(nonshooter_probs):,}")
print(f"Mean P(GSR): {nonshooter_probs.mean():.6f}")
print(f"Median P(GSR): {np.median(nonshooter_probs):.6f}")
print(f"P(GSR) < 0.1: {(nonshooter_probs < 0.1).mean():.2%}")
print(f"P(GSR) > 0.5: {(nonshooter_probs > 0.5).mean():.2%}")

print(f"\nShooter:")
print(f"N: {len(shooter_probs):,}")
print(f"Mean P(GSR): {shooter_probs.mean():.6f}")
print(f"Median P(GSR): {np.median(shooter_probs):.6f}")
print(f"P(GSR) < 0.1: {(shooter_probs < 0.1).mean():.2%}")
print(f"P(GSR) > 0.5: {(shooter_probs > 0.5).mean():.2%}")

print(f"\nContrast:")
print(f"Non-shooter GSR rate (0.5): {(nonshooter_probs >= 0.5).mean():.4%}")
print(f"Shooter GSR rate (0.5): {(shooter_probs >= 0.5).mean():.2%}")
print(
    f"Ratio: {(shooter_probs >= 0.5).mean() / max((nonshooter_probs >= 0.5).mean(), 1e-10):.1f}x"
)

Summary stats

Non-shooter:
N: 95,447
Mean P(GSR): 0.726836
Median P(GSR): 0.999507
P(GSR) < 0.1: 24.38%
P(GSR) > 0.5: 73.68%

Shooter:
N: 40,000
Mean P(GSR): 0.629867
Median P(GSR): 0.995922
P(GSR) < 0.1: 34.58%
P(GSR) > 0.5: 63.44%

Contrast:
Non-shooter GSR rate (0.5): 73.6828%
Shooter GSR rate (0.5): 63.44%
Ratio: 0.9x


## 8. Key findings 

The model is classifying the majority of ALL NIST particles as GSR regardless of source type. Brake dust is worse (81%) than fireworks (65%), but both are catastrophic. Brake dust triggering more false positives than fireworks makes sense since brake dust contains iron, calcium, and other metals that overlap with GSR elemental profiles more than firework residue does.

Bottom line though: This is a genuine cross-lab generalization failure driven by instrument calibration and measurement methodology differences. The model learned NFI-specific compositional distributions that don't transfer to NIST data. 

The MLP achieves near-perfect performance on NFI data. However, cross-dataset validation against NIST revealed a critical limitation. The model does not generalize across laboratories. This is driven by fundamental differences in measurement methodology, not model architecture. This finding has direct implications for deployment: any forensic GSR classification model must be validated on the target lab's data before use.